# Deliverable 2 -- Prompting Mejorado con Few-Shot y Post-procesamiento

In [32]:
import json
import re
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODELOS = {
    "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",              # ~2GB (baseline)
    "qwen4B": "Qwen/Qwen3-4B-Instruct-2507",                       # ~8-10GB (mejor candidato)
    "mistral7B": "mistralai/Mistral-7B-Instruct-v0.3",              # ~14GB (robusto)
}

MODELO_ACTUAL = "qwen"  # Cambia a "qwen" para usar este modelo
MODELO_ID = MODELOS[MODELO_ACTUAL]
FECHA_REFERENCIA = "2026-09-01"
DIA_REFERENCIA = "martes"

print(f"Modelo: {MODELO_ID}")
print(f"Fecha de referencia: {DIA_REFERENCIA} {FECHA_REFERENCIA}")

Modelo: Qwen/Qwen2-0.5B-Instruct
Fecha de referencia: martes 2026-09-01


In [33]:
# ============================================================
# CARGA O GENERACION DEL BENCHMARK
# ============================================================

BENCHMARK_FILE = "/content/frases (1).json"

if os.path.exists(BENCHMARK_FILE):
    with open(BENCHMARK_FILE, "r", encoding="utf-8") as f:
        benchmark = json.load(f)
    print(f"Benchmark cargado: {len(benchmark['frases'])} frases")
else:
  print("no lo identifica")

Benchmark cargado: 45 frases


In [34]:
# ============================================================
# CARGA DEL MODELO
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODELO_ID, torch_dtype=torch.float16, device_map="auto"
)
print(f"Modelo cargado en: {model.device}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Modelo cargado en: cpu


In [35]:
# ============================================================
# SYSTEM PROMPT: Reglas + Few-Shot (CORREGIDO)
# ============================================================

SYSTEM_PROMPT = """Eres un extractor de datos de calendario.

REGLAS:
1. HOY ES {dia} {fecha}. Calcula fechas relativas desde esta fecha.
2. Si un campo NO se menciona, usa null (o [] para participantes). NUNCA inventes.
3. titulo: solo la accion ("Reunión", "Cita", "Meeting", "Call"). Sin fechas ni horas.
4. fecha: YYYY-MM-DD. hora_inicio: HH:MM 24h.
5. duracion_minutos: entero (1 hora=60). Si no hay = null.
6. participantes: lista de nombres reales. Si no hay = [].
7. ubicacion: donde se realiza. Si no se menciona = null.
8. recordatorio_minutos: entero. Si no se menciona = null.
9. zona_horaria: "America/Santiago" salvo que se especifique otra.

EJEMPLOS:

Frase: "Reunión con Ana el viernes a las 10am"
{{"titulo": "Reunión", "fecha": "2026-09-04", "hora_inicio": "10:00", "duracion_minutos": null, "participantes": ["Ana"], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "Reunión de 2 horas el 15/09 a las 9:30 en la sala de conferencias, recordar 30 min"
{{"titulo": "Reunión", "fecha": "2026-09-15", "hora_inicio": "09:30", "duracion_minutos": 120, "participantes": [], "ubicacion": "Sala de conferencias", "recordatorio_minutos": 30, "zona_horaria": "America/Santiago"}}

Frase: "Reunión"
{{"titulo": "Reunión", "fecha": null, "hora_inicio": null, "duracion_minutos": null, "participantes": [], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "Meeting on Friday at 10am"
{{"titulo": "Meeting", "fecha": "2026-09-04", "hora_inicio": "10:00", "duracion_minutos": null, "participantes": [], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}"""

def construir_prompt(frase_texto):
    # Ahora format solo reemplazará {dia} y {fecha} porque los demás tienen doble llave
    system = SYSTEM_PROMPT.format(dia=DIA_REFERENCIA, fecha=FECHA_REFERENCIA)
    user_msg = f'Frase: "{frase_texto}"\n\nRespuesta (SOLO JSON):'
    if MODELO_ACTUAL == "qwen":
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user_msg}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        return f"<|system|>\n{system}\n<|user|>\n{user_msg}\n<|assistant|>\n"

print("Prompt configurado correctamente con llaves escapadas")

Prompt configurado correctamente con llaves escapadas


In [36]:
# ============================================================
# POST-PROCESAMIENTO
# ============================================================

def post_procesar(json_salida, frase_texto):
    if not isinstance(json_salida, dict) or "error" in json_salida:
        return json_salida
    r = json_salida.copy()

    # Corregir anho
    f = r.get("fecha")
    if isinstance(f, str):
        m = re.match(r"(\d{4})", f)
        if m and int(m.group(1)) not in (2026, 2027):
            r["fecha"] = f.replace(m.group(1), "2026", 1)

    # Corregir hora
    h = r.get("hora_inicio")
    if isinstance(h, str):
        hm = re.search(r"(\d{1,2}):(\d{2})", h)
        if hm:
            r["hora_inicio"] = f"{int(hm.group(1)):02d}:{int(hm.group(2)):02d}"

    # Limpiar participantes
    p = r.get("participantes", [])
    if isinstance(p, list):
        r["participantes"] = [x for x in p if isinstance(x, str) and not x.startswith("<") and len(x.strip()) > 0 and not x.isdigit()]

    # Numerico
    for campo in ["duracion_minutos", "recordatorio_minutos"]:
        val = r.get(campo)
        if val is not None and not isinstance(val, (int, float)):
            nm = re.search(r"(\d+)", str(val))
            r[campo] = int(nm.group(1)) if nm else None

    # Titulo limpio
    t = r.get("titulo", "")
    if isinstance(t, str) and (len(t) > 50 or re.search(r"\d{1,2}[/-]\d{1,2}", t)):
        for palabra in ["Reuni\u00f3n", "Cita", "Meeting", "Call", "Appointment"]:
            if palabra.lower() in t.lower():
                r["titulo"] = palabra
                break
    return r

print("Post-procesador listo")

Post-procesador listo


In [ ]:
# ============================================================
# EJECUCION
# ============================================================
resultados = []

for frase in benchmark["frases"]:
    prompt = construir_prompt(frase["texto"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=256, temperature=0.01,
            do_sample=True, top_p=0.95, pad_token_id=tokenizer.eos_token_id
        )

    respuesta = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True).strip()
    json_match = re.search(r"\{.*\}", respuesta, re.DOTALL)
    json_salida = {"error": "No JSON detected"}
    if json_match:
        try:
            json_salida = json.loads(json_match.group())
        except json.JSONDecodeError:
            json_salida = {"error": "Invalid JSON", "raw": respuesta[:200]}

    json_salida = post_procesar(json_salida, frase["texto"])
    resultados.append({"id": frase["id"], "texto": frase["texto"], "resultado_modelo": json_salida})
    print(f"Frase {frase['id']:2d}: {'OK' if 'error' not in json_salida else 'ERROR'}")

output_file = f"resultados_{MODELO_ACTUAL}_mejorado.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)
print(f"\nGuardado: {output_file}")

Frase  1: OK
Frase  2: OK


In [ ]:
# ============================================================
# EVALUACION vs GROUND TRUTH CON REPORTE DESCARGABLE (CSV)
# ============================================================
from google.colab import files
import csv

CAMPOS = ["titulo", "fecha", "hora_inicio", "duracion_minutos",
          "participantes", "ubicacion", "recordatorio_minutos", "zona_horaria"]

correctos = 0
errores_campo = {c: 0 for c in CAMPOS}
detalles = []  # Aquí guardamos cada registro para el CSV

for r in resultados:
    # Buscar el ground truth correspondiente
    gt = next((f for f in benchmark["frases"] if f["id"] == r["id"]), None)
    if gt is None:
        continue
    gt = gt["ground_truth"]
    pred = r["resultado_modelo"]

    # Si el modelo devolvió un error, marcamos todos los campos como incorrectos
    if "error" in pred:
        for c in CAMPOS:
            errores_campo[c] += 1
        detalles.append({
            "id": r["id"],
            "texto": r["texto"],
            "exacto": False,
            **{f"{c}_gt": gt.get(c) for c in CAMPOS},
            **{f"{c}_pred": "ERROR_MODELO" for c in CAMPOS},
            **{f"{c}_ok": False for c in CAMPOS}
        })
        continue

    ok_general = True
    row_detail = {
        "id": r["id"],
        "texto": r["texto"],
        "exacto": True
    }

    for c in CAMPOS:
        esp = gt.get(c)
        opt = pred.get(c)
        field_ok = True  # Asumimos que es correcto

        # Lógica de comparación (igual que antes)
        if esp is None and opt is None:
            pass  # Ambos nulos -> correcto
        elif esp is None and opt is not None:
            errores_campo[c] += 1
            ok_general = False
            field_ok = False
        elif esp is not None and opt is None:
            errores_campo[c] += 1
            ok_general = False
            field_ok = False
        elif isinstance(esp, list):
            # Comparar listas (participantes) ignorando mayúsculas
            pred_list = [str(x).lower() for x in (opt or [])]
            gt_list = [str(x).lower() for x in esp]
            if set(gt_list) != set(pred_list):
                errores_campo[c] += 1
                ok_general = False
                field_ok = False
        else:
            # Comparación simple de strings
            if str(esp).lower() != str(opt).lower():
                errores_campo[c] += 1
                ok_general = False
                field_ok = False

        # Guardamos los valores para el reporte
        row_detail[f"{c}_gt"] = esp
        row_detail[f"{c}_pred"] = opt
        row_detail[f"{c}_ok"] = field_ok

    row_detail["exacto"] = ok_general
    if ok_general:
        correctos += 1
    detalles.append(row_detail)

# ---- Mostrar resultados en consola ----
total = len(resultados)
print(f"\n{'='*50}")
print(f"  {MODELO_ACTUAL.upper()} MEJORADO")
print(f"  Precision: {correctos}/{total} ({correctos/total*100:.1f}%)")
print(f"{'='*50}")
for c in CAMPOS:
    ok = total - errores_campo[c]
    print(f"  {c:25s}: {ok}/{total} ({ok/total*100:.1f}%)")

# ---- GENERAR Y DESCARGAR REPORTE CSV ----
reporte_file = f"reporte_{MODELO_ACTUAL}.csv"

with open(reporte_file, "w", newline='', encoding='utf-8') as f:
    # Construir los encabezados dinámicamente
    base_headers = ["id", "texto", "exacto"]
    field_headers = []
    for c in CAMPOS:
        field_headers.extend([f"{c}_gt", f"{c}_pred", f"{c}_ok"])
    headers = base_headers + field_headers

    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()

    # Escribir cada detalle
    for row in detalles:
        # Asegurar que todas las claves existen (por si falta alguna)
        write_row = {h: row.get(h, "") for h in headers}
        writer.writerow(write_row)

    # ---- Añadir resumen al final del CSV ----
    writer.writerow({h: "" for h in headers})  # Línea en blanco
    writer.writerow({"id": "RESUMEN", "texto": f"Exactitud General: {correctos}/{total} ({correctos/total*100:.1f}%)"})
    for c in CAMPOS:
        ok = total - errores_campo[c]
        writer.writerow({"id": f"Campo_{c}", "texto": f"{ok}/{total} ({ok/total*100:.1f}%)"})

# Descargar el archivo a tu máquina
files.download(reporte_file)
print(f"\n✅ Reporte descargado: {reporte_file}")